# [수학 Retreat #5] 도형의 성질: 삼각형의 외심·내심과 사각형의 진화

이 주피터 노트북은 중학교 2학년 과정의 **도형의 성질** 단원을 파이썬 2D 그래픽 시각화를 통해 직관적으로 탐구하는 실습 교재입니다.
기하학적 도형 속에 내재된 대칭, 균형, 그리고 포함 관계의 아름다운 질서를 다룹니다.

### 🟢 본 실습에서 다룰 두 가지 주제:
1. **삼각형의 중심 (`triangle_centers`)**: 삼각형의 세 꼭짓점을 드래그하여 형태를 바꿀 때, 외심(외접원의 중심)과 내심(내접원의 중심)이 실시간으로 이동하고 안팎을 둘러싼 원이 삼각형을 포용하는 작도 시뮬레이션.
2. **사각형의 진화 (`quadrilateral_hierarchy`)**: 사각형의 평행 조건, 길이 조건, 직각 조건이 누적됨에 따라, 일반 사각형이 사다리꼴 $\rightarrow$ 평행사변형 $\rightarrow$ 직사각형/마름모 $\rightarrow$ 정사각형으로 부드럽게 **모핑(Morphing)**하며 형태가 진화하는 계층적 관계 학습 시뮬레이션.

**⚠️ 안내:** 본 파일은 파이썬 초보자도 코드 한 줄 한 줄의 기하학적·수학적 연산 원리를 이해할 수 있도록 **세포 단위의 멀티셀**로 쪼개어 구성했으며 상세한 한글 주석을 부착했습니다.


## 1. 환경 준비 및 필수 라이브러리 설치
본 실습에서는 기하 좌표 계산을 위해 `numpy`, 2D 동적 그래픽과 모핑 애니메이션을 위해 `plotly`, 그리고 제어 위젯을 조립하기 위해 `ipywidgets`를 활용합니다.
아래 셀을 실행하여 라이브러리를 먼저 준비합니다.


In [1]:
# !pip install 명령어는 주피터 노트북 내에서 터미널의 패키지 매니저를 호출하여 패키지를 설치하게 하는 매직 명령어입니다.
# -q (quiet) 옵션을 붙여 설치 메시지를 생략하고 설치합니다.
%pip install -q numpy plotly ipywidgets


Note: you may need to restart the kernel to use updated packages.


### 1.2 모듈 로드 (Import)
기하 작도 좌표 연산과 인터랙티브 대시보드 구동에 필요한 모듈들을 메모리에 적재합니다.


In [2]:
# numpy: 점의 좌표 표현, 선분 길이 계산, 삼각함수 좌표 변환 등의 대수적 처리를 돕는 파이썬의 핵심 수학 라이브러리입니다.
import numpy as np

# plotly.graph_objects: 동적으로 꼭짓점과 외접원/내접원이 갱신되어 렌더링되게 돕는 2D 그래프 플로팅 도구입니다.
import plotly.graph_objects as go

# ipywidgets: 꼭짓점의 좌표 슬라이더 및 포함 조건 체크박스 등의 제어반 컴포넌트를 구성하기 위한 위젯 라이브러리입니다.
import ipywidgets as widgets

# IPython.display: 정보 리포트 대시보드를 주피터 노트북 내에 HTML 박스로 그리기 위해 로드합니다.
from IPython.display import display, HTML


---
## 2. 실습 1: 삼각형의 두 중심 - 외심(Circumcenter)과 내심(Incenter)
삼각형은 5가지의 특이한 중심(오심)을 가지며, 그 중 대표적인 두 개가 **외심**과 **내심**입니다.

1. **외심(Circumcenter)**:
   - 정의: 세 변의 **수직이등분선의 교점**입니다.
   - 성질: 외심에서 삼각형의 **세 꼭짓점에 이르는 거리가 같습니다** ($OA = OB = OC$). 이 거리가 외접원의 반지름 $R$이 됩니다.
   - 철학적 대변: 개별적인 존재(꼭짓점)들을 고르게 보살피는 **'외부적 포용'**을 뜻합니다.
2. **내심(Incenter)**:
   - 정의: 세 내각의 **이등분선의 교점**입니다.
   - 성질: 내심에서 삼각형의 **세 변에 이르는 수직 거리가 같습니다** ($ID = IE = IF$). 이 거리가 내접원의 반지름 $r$이 됩니다.
   - 철학적 대변: 외부 자극에 흔들리지 않고 경계를 수호하는 **'내부적 정체성'**을 뜻합니다.

꼭짓점 A, B, C의 좌표로부터 외심과 내심의 중심점 좌표 및 내/외접원의 반지름을 수학적 행렬식에 맞추어 연산하는 함수를 정의합니다.


In [3]:
def calculate_triangle_centers(A, B, C):
    """
    세 꼭짓점 A, B, C의 [x, y] 좌표를 입력받아 외심(O), 외접원 반지름(R), 내심(I), 내접원 반지름(r)을 계산합니다.
    """
    x1, y1 = A
    x2, y2 = B
    x3, y3 = C
    
    # 1. 외심 (Circumcenter) 연산
    # 수직이등분선의 연립방정식을 행렬식(Cramer's rule) 형태로 풀어내어 수치적 오차 없이 외심 좌표를 구합니다.
    D = 2 * (x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
    
    # 만약 세 점이 일직선 상에 놓여 있어 삼각형이 성립하지 않는 경우 예외 처리
    if abs(D) < 1e-9:
        return None
        
    # 외심 O(xo, yo) 좌표 공식
    xo = ((x1**2 + y1**2) * (y2 - y3) + (x2**2 + y2**2) * (y3 - y1) + (x3**2 + y3**2) * (y1 - y2)) / D
    yo = ((x1**2 + y1**2) * (x3 - x2) + (x2**2 + y2**2) * (x1 - x3) + (x3**2 + y3**2) * (x2 - x1)) / D
    circumcenter = np.array([xo, yo])
    
    # 외접원의 반지름 R: 외심에서 꼭짓점 A(또는 B, C)까지의 피타고라스 거리
    R = np.sqrt((xo - x1)**2 + (yo - y1)**2)
    
    # 2. 내심 (Incenter) 연산
    # 세 변의 길이 a(BC의 길이), b(CA의 길이), c(AB의 길이)를 산출합니다.
    a = np.sqrt((x2 - x3)**2 + (y2 - y3)**2)
    b = np.sqrt((x1 - x3)**2 + (y1 - y3)**2)
    c = np.sqrt((x1 - x2)**2 + (y1 - y2)**2)
    
    # 내심 I(xi, yi) 좌표 공식: 세 꼭짓점 좌표를 대변의 길이 비율로 가중평균한 기하 좌표입니다.
    xi = (a * x1 + b * x2 + c * x3) / (a + b + c)
    yi = (a * y1 + b * y2 + c * y3) / (a + b + c)
    incenter = np.array([xi, yi])
    
    # 내접원의 반지름 r: 2 * 삼각형의 면적 / 삼각형의 둘레(a+b+c)
    # 신발끈 공식(사선 공식)을 사용해 삼각형의 면적을 구합니다.
    area = 0.5 * abs(x1 * (y2 - y3) + x2 * (y3 - y1) + x3 * (y1 - y2))
    r = (2 * area) / (a + b + c)
    
    return circumcenter, R, incenter, r


### 2.2 실시간 삼각형 정보 리포트 대시보드 설계
삼각형 꼭짓점이 조절될 때마다 현재 삼각형의 모양 분류(예: 예각, 직각, 둔각) 및 외심/내심의 특징적인 위치 상태를 나타내 주는 대시보드를 생성하는 HTML 함수를 작성합니다.


In [4]:
def build_triangle_info_html(A, B, C, R, r, O, I):
    """
    현재 실시간 삼각형 꼭짓점에 상응하는 기하 정보를 예쁜 대시보드 창으로 구성합니다.
    """
    # 세 변의 길이 연산
    a = np.sqrt((B[0] - C[0])**2 + (B[1] - C[1])**2)
    b = np.sqrt((A[0] - C[0])**2 + (A[1] - C[1])**2)
    c = np.sqrt((A[0] - B[0])**2 + (A[1] - B[1])**2)
    
    # 각 변의 제곱값 정렬을 통한 삼각형 분류 (피타고라스 이용)
    sides_sq = sorted([a**2, b**2, c**2])
    # 최장변의 제곱이 나머지 두 변의 제곱 합과 같으면 직각, 더 크면 둔각, 작으면 예각삼각형
    if abs(sides_sq[2] - (sides_sq[0] + sides_sq[1])) < 1e-5:
        tri_type = "직각삼각형 (Right Triangle)"
    elif sides_sq[2] > (sides_sq[0] + sides_sq[1]):
        tri_type = "둔각삼각형 (Obtuse Triangle)"
    else:
        tri_type = "예각삼각형 (Acute Triangle)"
        
    html_template = f"""
    <div style='background: rgba(255, 255, 255, 0.85);
                backdrop-filter: blur(8px);
                border: 1.5px solid rgba(11, 121, 208, 0.25);
                border-radius: 14px; padding: 20px; max-width: 600px;
                font-family: sans-serif; box-shadow: 0 4px 15px rgba(0,0,0,0.06);'>
        <h4 style='margin: 0 0 10px 0; color: #0B79D0;'>🔺 삼각형 기하 상태 분석 리포트</h4>
        <table style='width: 100%; border-collapse: collapse; font-size: 0.9em;'>
            <tr style='border-bottom: 1px solid rgba(0,0,0,0.05);'>
                <td style='padding: 8px 0; font-weight: bold;'>삼각형 분류</td>
                <td style='padding: 8px 0; color: #0B79D0; font-weight: bold;'>{tri_type}</td>
            </tr>
            <tr style='border-bottom: 1px solid rgba(0,0,0,0.05);'>
                <td style='padding: 8px 0; font-weight: bold;'>외심의 위치 및 외접원 반지름 (R)</td>
                <td style='padding: 8px 0;'>O({O[0]:.2f}, {O[1]:.2f}) | <b>R = {R:.2f}</b></td>
            </tr>
            <tr style='border-bottom: 1px solid rgba(0,0,0,0.05);'>
                <td style='padding: 8px 0; font-weight: bold;'>내심의 위치 및 내접원 반지름 (r)</td>
                <td style='padding: 8px 0;'>I({I[0]:.2f}, {I[1]:.2f}) | <b>r = {r:.2f}</b></td>
            </tr>
        </table>
        <div style='margin-top: 10px; font-size: 0.85em; color: #666; font-style: italic;'>
            💡 <b>관찰 포인트:</b> 둔각삼각형일 때 외심(O)은 삼각형의 완전히 <b>외부 영역</b>으로 탈출하며, 직각삼각형일 때 외심은 정확히 <b>빗변의 중점</b> 위에 위치하게 됩니다. 반면, 내접원을 품는 내심(I)은 언제나 흔들림 없이 삼각형의 <b>내부 영역</b> 중심에 견고히 고정되어 있습니다.
        </div>
    </div>
    """
    return html_template


### 2.3 Plotly 2D 동적 기하 작도 렌더링
삼각형의 둘레선, 외접원 및 내접원(삼각함수 매개변수 표현 활용), 수직이등분선과 각이등분선의 연장 지시선, 그리고 외심(O)과 내심(I) 점 마커를 캔버스에 그리는 함수를 구축합니다.


In [5]:
def draw_triangle_simulation_scene(A_x, A_y, B_x, B_y, C_x, C_y):
    """
    꼭짓점 슬라이더 신호들에 맞춰 삼각형 기하 작도를 2D 상에 플로팅합니다.
    """
    A = np.array([A_x, A_y])
    B = np.array([B_x, B_y])
    C = np.array([C_x, C_y])
    
    # 1. 외심/내심 연산
    center_results = calculate_triangle_centers(A, B, C)
    if center_results is None:
        print("삼각형의 면적이 형성되지 않습니다. 꼭짓점 위치를 재조정해 주세요.")
        return
        
    O, R, I, r = center_results
    
    # 2. HTML 분석 리포트 화면 표출
    display(HTML(build_triangle_info_html(A, B, C, R, r, O, I)))
    
    # 3. 원 궤적 생성을 위한 삼각함수 매개 각도 배열 생성 (0도 ~ 360도)
    angles = np.linspace(0, 2*np.pi, 200)
    
    # 4. 외접원(Circumcircle)의 원주 좌표 연산: O(xo, yo) 기준 반경 R
    circum_x = O[0] + R * np.cos(angles)
    circum_y = O[1] + R * np.sin(angles)
    
    # 5. 내접원(Incircle)의 원주 좌표 연산: I(xi, yi) 기준 반경 r
    in_x = I[0] + r * np.cos(angles)
    in_y = I[1] + r * np.sin(angles)
    
    # 6. Plotly 트레이스 빌드
    traces = []
    
    # 1) 삼각형 테두리 선분 (꼭짓점 A -> B -> C -> A 회폐쇄)
    traces.append(go.Scatter(
        x=[A_x, B_x, C_x, A_x],
        y=[A_y, B_y, C_y, A_y],
        mode='lines+markers',
        line=dict(color='#1E2937', width=4.5), # 짙은 먹색의 튼튼한 삼각형 벽선
        marker=dict(size=12, color='#1E2937'),
        name='삼각형 ABC'
    ))
    
    # 2) 외접원 플롯 (블루 그라데이션 컨셉의 파란 원선)
    traces.append(go.Scatter(
        x=circum_x, y=circum_y,
        mode='lines',
        line=dict(color='#0B79D0', width=2.5), # 브랜드 블루
        name='외접원 (Circumcircle)'
    ))
    
    # 3) 내접원 플롯 (브랜드 그린 컬러의 초록 원선)
    traces.append(go.Scatter(
        x=in_x, y=in_y,
        mode='lines',
        line=dict(color='#2FA85D', width=2.5), # 브랜드 그린
        name='내접원 (Incircle)'
    ))
    
    # 4) 외심 O 점 마커 플롯 (파란색 별표)
    traces.append(go.Scatter(
        x=[O[0]], y=[O[1]],
        mode='markers+text',
        marker=dict(size=14, color='#0B79D0', symbol='star', line=dict(color='white', width=1)),
        text=['외심 (O)'], textposition='top right',
        name='외심 (Circumcenter)'
    ))
    
    # 5) 내심 I 점 마커 플롯 (초록색 별표)
    traces.append(go.Scatter(
        x=[I[0]], y=[I[1]],
        mode='markers+text',
        marker=dict(size=14, color='#2FA85D', symbol='star', line=dict(color='white', width=1)),
        text=['내심 (I)'], textposition='bottom left',
        name='내심 (Incenter)'
    ))
    
    # 레이아웃 정의 (동등한 등비율 x-y 격자 눈금 확보 필수)
    layout = go.Layout(
        title=dict(text='<b>삼각형 외심 & 내심 기하 작도 시뮬레이션</b>', x=0.5),
        xaxis=dict(range=[-10, 10], gridcolor='#F1F5F9', zeroline=True, zerolinecolor='#CBD5E1'),
        yaxis=dict(range=[-10, 10], gridcolor='#F1F5F9', scaleanchor='x', scaleratio=1.0, zeroline=True, zerolinecolor='#CBD5E1'),
        plot_bgcolor='white',
        width=650, height=600,
        showlegend=True,
        legend=dict(x=0.02, y=0.98, bordercolor='#E2E8F0', borderwidth=1)
    )
    
    fig = go.Figure(data=traces, layout=layout)
    fig.show()


### 2.4 인터랙티브 삼각형 조작 대시보드 구동
삼각형의 세 꼭짓점 A, B, C의 x, y 좌표들을 독립적으로 조절하는 6개의 슬라이더를 세로형 패널로 묶어 기동합니다.
슬라이더를 조작하며 삼각형의 모양을 예각, 직각, 둔각으로 변형시켜 외심과 내심의 움직임 양상을 주의 깊게 통찰해 보세요.


In [6]:
# 꼭짓점 A(x, y) 제어 슬라이더
slider_Ax = widgets.FloatSlider(value=0.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 A_x:')
slider_Ay = widgets.FloatSlider(value=6.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 A_y:')

# 꼭짓점 B(x, y) 제어 슬라이더
slider_Bx = widgets.FloatSlider(value=-6.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 B_x:')
slider_By = widgets.FloatSlider(value=-3.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 B_y:')

# 꼭짓점 C(x, y) 제어 슬라이더
slider_Cx = widgets.FloatSlider(value=5.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 C_x:')
slider_Cy = widgets.FloatSlider(value=-2.0, min=-7.0, max=7.0, step=0.2, description='꼭짓점 C_y:')

# 꼭짓점별로 컨트롤 팩을 나누어 정렬합니다.
controls_box = widgets.VBox([
    widgets.HTML("<h4 style='color:#0B79D0; margin:0;'> 꼭짓점 A 조작</h4>"), slider_Ax, slider_Ay,
    widgets.HTML("<h4 style='color:#2FA85D; margin:0;'> 꼭짓점 B 조작</h4>"), slider_Bx, slider_By,
    widgets.HTML("<h4 style='color:#E11D48; margin:0;'> 꼭짓점 C 조작</h4>"), slider_Cx, slider_Cy
])

# 위젯 변수들을 렌더러 함수에 대칭 결합
output_scene = widgets.interactive_output(
    draw_triangle_simulation_scene,
    {
        'A_x': slider_Ax, 'A_y': slider_Ay,
        'B_x': slider_Bx, 'B_y': slider_By,
        'C_x': slider_Cx, 'C_y': slider_Cy
    }
)

# 세로 제어 패널(왼쪽)과 2D 작도 창(오른쪽) 가로 배치
display(widgets.HBox([controls_box, output_scene]))


---
## 3. 실습 2: 사각형의 진화와 포함 관계 모핑 시뮬레이터 (`quadrilateral_hierarchy`)
사각형은 조건이 추가(누적)됨에 따라 점진적으로 계층적 진화를 거듭합니다.

### 사각형 계층 구조의 조건 누적 관계:
1. **일반 사각형**: 아무런 특수한 조건이 없는 일반적인 4각 다각형.
2. **사다리꼴 (Trapezoid)**: 한 쌍의 대변이 평행함 (평행 조건 1단계).
3. **평행사변형 (Parallelogram)**: 두 쌍의 대변이 평행함 (평행 조건 2단계 완성).
4. **직사각형 (Rectangle)**: 네 내각의 크기가 같음 ($90^\circ$, 각의 조건 완성).
5. **마름모 (Rhombus)**: 네 변의 길이가 같음 (변의 조건 완성).
6. **정사각형 (Square)**: 네 내각이 같고 네 변의 길이도 같은 완벽한 상태 (각과 변의 조건 완전 융합).

각 형태에 상응하는 고유 기하 좌표 점들을 정의하고, 조건 선택에 따라 현재 사각형의 점들이 다음 사각형의 점들로 부드럽게 이동(Morphing)하는 선형 보간 프레임을 설계하겠습니다.


In [10]:
# 사각형의 6개 형태별 고유 기준 2D 꼭짓점 4개의 좌표 목록 (x, y)
# 닫힌 사각형 형태의 표현을 위해 마지막에 첫 번째 점을 덧대어 5개의 점으로 매핑합니다.
quadrilateral_states = {
    '1. 일반 사각형': np.array([[0.0, 0.0], [5.0, 0.8], [4.0, 4.0], [0.5, 3.0], [0.0, 0.0]]),
    '2. 사다리꼴': np.array([[0.0, 0.0], [5.0, 0.0], [3.5, 3.0], [1.5, 3.0], [0.0, 0.0]]),
    '3. 평행사변형': np.array([[0.0, 0.0], [4.0, 0.0], [5.2, 3.0], [1.2, 3.0], [0.0, 0.0]]),
    '4. 직사각형': np.array([[0.0, 0.0], [5.0, 0.0], [5.0, 3.0], [0.0, 3.0], [0.0, 0.0]]),
    '5. 마름모': np.array([[0.0, 0.0], [3.5, 0.0], [5.25, 3.0], [1.75, 3.0], [0.0, 0.0]]), # 네 변의 길이가 3.5인 형태
    '6. 정사각형': np.array([[0.0, 0.0], [3.5, 0.0], [3.5, 3.5], [0.0, 3.5], [0.0, 0.0]])
}

# 이전 선택 상태의 점들을 메모리에 캐싱해 두어 매끄러운 징검다리 선형 보간을 하기 위한 글로벌 변수
global_last_points = quadrilateral_states['1. 일반 사각형'].copy()


### 3.2 사각형 트랜스포메이션(Morphing) 및 프레임 빌더 구성
사용자가 사각형 종류를 클릭하면, 캐시된 이전 꼭짓점 행렬 $P_{start}$부터 목표 꼭짓점 행렬 $P_{end}$까지 시간 계수 $t$ ($0 \le t \le 1$)에 따라,
$$P(t) = (1 - t)P_{start} + t P_{end}$$
의 보간법을 적용해 12개의 애니메이션 보간 프레임을 생성해 Plotly의 `frames` 모듈에 공급합니다.


In [11]:
def play_quadrilateral_morphing_animation(target_state):
    """
    선택한 사각형 상태(target_state)로 꼭짓점들이 부드럽게 미끄러지듯 이동하는 모핑 애니메이션을 생성합니다.
    """
    global global_last_points
    
    # 1. 시작점과 목표점 설정
    start_pts = global_last_points
    end_pts = quadrilateral_states[target_state]
    
    # 2. 보간 프레임 수 정의 (12 프레임을 사용해 자연스러움 연출)
    frame_count = 12
    time_steps = np.linspace(0.0, 1.0, frame_count)
    
    # 3. 애니메이션의 기본 토대가 되는 첫 번째 뼈대(Base Figure) 구성
    base_trace = go.Scatter(
        x=start_pts[:, 0], y=start_pts[:, 1],
        mode='lines+markers',
        line=dict(color='#0B79D0', width=4), # 사각형 선분은 브랜드 블루 컬러
        marker=dict(size=10, color='#2FA85D'), # 꼭짓점들은 브랜드 그린 마커
        name='사각형 외곽선'
    )
    
    # 4. 각 시간 스텝별 프레임 리스트 조립
    animation_frames = []
    for step in time_steps:
        # 선형 보간 공식 적용: t가 0일 땐 start_pts, 1일 땐 end_pts로 선형 변환
        interp_pts = (1 - step) * start_pts + step * end_pts
        
        # 프레임 딕셔너리를 작성해 어펜드
        animation_frames.append(go.Frame(
            data=[go.Scatter(
                x=interp_pts[:, 0],
                y=interp_pts[:, 1]
            )],
            name=f'f_{step}'
        ))
        
    # 5. 애니메이션 재생 및 일시정지 제어용 UI 버튼 부착
    updatemenus_config = [dict(
        type='buttons',
        showactive=False,
        buttons=[
            dict(
                label='▶ 진화 재생 (Play)',
                method='animate',
                args=[None, dict(frame=dict(duration=35, redraw=False), fromcurrent=True)]
            ),
            dict(
                label='|| 일시정지 (Pause)',
                method='animate',
                args=[[None], dict(frame=dict(duration=0, redraw=False), mode='immediate')]
            )
        ],
        direction='left',
        pad=dict(r=10, t=10),
        x=0.1, y=-0.15
    )]
    
    # 6. 애니메이션 피규어 조립
    fig = go.Figure(
        data=[base_trace],
        layout=go.Layout(
            title=dict(text=f'<b>사각형 진화 모핑 (현재 모형: {target_state})</b>', x=0.5),
            xaxis=dict(range=[-1.0, 7.0], gridcolor='#F1F5F9', zeroline=True, zerolinecolor='#CBD5E1'),
            yaxis=dict(range=[-1.0, 7.0], gridcolor='#F1F5F9', scaleanchor='x', scaleratio=1.0, zeroline=True, zerolinecolor='#CBD5E1'),
            plot_bgcolor='white',
            width=600, height=520,
            updatemenus=updatemenus_config,
            margin=dict(l=40, r=40, b=80, t=60)
        ),
        frames=animation_frames
    )
    
    # 7. 다음 상태 보간을 위해 이전 점의 위치를 현재의 목표 상태 점으로 업데이트 해 둡니다.
    global_last_points = end_pts.copy()
    
    # 그래프 출력
    fig.show()


### 3.3 사각형 진화 제어반 구동
사각형의 6가지 상태를 선택할 수 있는 라디오 버튼 위젯을 생성하고 모핑 렌더러 함수에 바인딩합니다.
라디오 버튼을 순서대로 클릭하고 **'▶ 진화 재생'** 버튼을 클릭하여, 선분들이 부드러운 애니메이션을 그리며 목표 형태로 정렬되는 사각형의 포함 관계 기하학을 체감해 보세요.


In [ ]:
# 사각형의 6가지 단계 선택 라디오 버튼 생성
quad_selector_widget = widgets.RadioButtons(
    options=['1. 일반 사각형', '2. 사다리꼴', '3. 평행사변형', '4. 직사각형', '5. 마름모', '6. 정사각형'],
    value='1. 일반 사각형',
    description='진화 상태:',
    style={'description_width': 'initial'}
)

# interact를 활용한 위젯 바인딩 기동
widgets.interact(play_quadrilateral_morphing_animation, target_state=quad_selector_widget);


interactive(children=(RadioButtons(description='진화 상태:', options=('1. 일반 사각형', '2. 사다리꼴', '3. 평행사변형', '4. 직사각형…

---
## 4. 깊이 있는 기하학적 사색을 위한 열린 질문 (Joshua를 위한 질문)

1. **외심의 흔들림과 내심의 견고함**:
   삼각형을 납작한 둔각삼각형으로 변형시킬 때, 외심(외부 포용)은 삼각형의 내부 울타리를 탈출하여 멀리 도망치는 경계의 불안정성을 보이는 반면, 내심(내부 정체성)은 항상 삼각형 안에서 안전하게 영역을 유지합니다.
   개인의 내면적 도덕성(내심)과, 세상을 아우르는 포용의 규칙(외심) 중에서 내 삶과 기업 운영에 우선시해야 할 기하학적 지향점은 어디에 위치해 있는지 사색해 보고 리더십적인 관점의 소회를 나눠 보세요.

2. **정사각형(Square)을 이루는 조건의 점진적 완성**:
   일반 사각형이 정사각형에 도달하기까지 평행성, 등각성, 등변성의 조건들이 차례로 누적(진화)되었습니다. 우리 인생이나 비즈니스가 미완성의 혼돈(일반 사각형)을 지나 명확한 질서와 완벽함(정사각형)에 이르기 위해, 현재 차례로 축적해 나가야 할 나만의 핵심적인 '조건'들은 어떤 것들이 있는지 사색해 보세요.
